# Mini-Batch Gradient Descent — Beginner Masterclass

**Main source:** [GeeksforGeeks — Mini-Batch Gradient Descent in Deep Learning](https://www.geeksforgeeks.org/deep-learning/mini-batch-gradient-descent-in-deep-learning/)

This Colab notebook explains mini-batch gradient descent from the ground up. It follows the source article's ideas—dividing data into batches, calculating batch gradients, updating parameters, tracking batch loss, selecting batch size, and using TensorFlow/PyTorch—then adds clear hand calculations and fair comparisons with full-batch Gradient Descent and Stochastic Gradient Descent (SGD).

## What you'll learn

1. What an epoch, mini-batch, iteration, gradient, and learning rate mean.
2. How to compute a mini-batch gradient by hand.
3. How mini-batch updates differ from batch GD and single-sample SGD.
4. How to implement mini-batch gradient descent using plain Python and NumPy.
5. How batch size and learning rate affect convergence.
6. How to train an actual model with PyTorch and TensorFlow/Keras.
7. Why you should plot both batch-level and epoch-level loss.
8. Common bugs, including handling the final smaller batch correctly.

## 1. What is mini-batch gradient descent?

Mini-batch gradient descent divides the training dataset into small groups called **mini-batches**. It computes a gradient using one mini-batch and then updates the model parameters.

\[
\boxed{\theta_{t+1}=\theta_t-\eta\nabla_\theta J_{B_t}(\theta_t)}
\]

- \(\theta_t\): model parameters at the current update.
- \(\eta\): learning rate.
- \(B_t\): the mini-batch used for this update.
- \(J_{B_t}\): average loss over that mini-batch.
- \(\nabla_\theta J_{B_t}\): gradient of the mini-batch loss.

The benefit is a compromise:
- Full-batch GD uses all training examples per update.
- Strict SGD uses one training example per update.
- Mini-batch GD uses a small subset, often 32, 64, or 128 examples.

Mini-batches can use GPUs efficiently and typically have less noisy gradients than single-example SGD, while updating more often than full-batch GD. This is a common practice in modern deep learning.

## 2. The key vocabulary

- **Sample / example:** one input and its target.
- **Batch size \(B\):** number of examples used to estimate one gradient.
- **Iteration / update step:** one optimizer update.
- **Epoch:** one complete pass through the training dataset.
- **Batch loss:** loss calculated on one mini-batch.
- **Epoch loss:** a summary loss over an entire dataset or the average of its batches.

If you have \(N=1000\) training examples and batch size \(B=100\), there are 10 updates per epoch.

\[
\text{updates per epoch}=\left\lceil\frac{N}{B}\right\rceil
\]

The ceiling matters when the final batch contains fewer than \(B\) samples. Frameworks commonly keep that final partial batch unless configured to drop it.

In [ ]:
N = 1000
batch_sizes = [1, 32, 64, 100, 128, 512, 1000]
rows = []
for B in batch_sizes:
    rows.append({
        "Dataset size N": N,
        "Batch size B": B,
        "Updates per epoch (ceil(N/B))": (N + B - 1) // B,
        "Final batch size": N - B * ((N - 1) // B)
    })
display(pd.DataFrame(rows) if "pd" in globals() else __import__("pandas").DataFrame(rows))

## 3. The three main variants compared

| Method | Examples used for each update | Main trade-off |
|---|---:|---|
| Batch Gradient Descent | Entire training set | Accurate, stable gradient estimate; expensive for large datasets |
| Stochastic Gradient Descent (SGD) | One example | Very frequent updates; noisy loss/gradient |
| Mini-Batch Gradient Descent | Small group, e.g. 32–128 | Balances computation, gradient noise, and update frequency |

The word **SGD** is sometimes used casually to describe mini-batch training too. In this notebook, “strict SGD” means a batch size of one.

## 4. Derive a mini-batch update for linear regression

We'll use the model:

\[
\hat y_i=mx_i+c
\]

with Mean Squared Error:

\[
J_B(m,c)=\frac1B\sum_{i=1}^{B}(mx_i+c-y_i)^2
\]

The mini-batch gradient is the average of the individual gradients:

\[
\frac{\partial J_B}{\partial m}
=\frac{2}{B}\sum_{i=1}^{B}(\hat y_i-y_i)x_i
\]

\[
\frac{\partial J_B}{\partial c}
=\frac{2}{B}\sum_{i=1}^{B}(\hat y_i-y_i)
\]

Then update both parameters:

\[
m\leftarrow m-\eta\frac{\partial J_B}{\partial m},
\quad
c\leftarrow c-\eta\frac{\partial J_B}{\partial c}.
\]

The important difference from strict SGD is that the mini-batch gradients are averaged over multiple samples before changing the parameters.

## 5. Worked calculation: one mini-batch update

Take two points:

\[
x=[1,2],\qquad y=[2,4]
\]

Start with \(m=0,c=0\), batch size \(B=2\), and learning rate \(\eta=0.1\).

**Step 1 — Predictions**

\[
\hat y=[0,0]
\]

**Step 2 — Errors \(\hat y-y\)**

\[
e=[-2,-4]
\]

**Step 3 — Mean squared error**

\[
J_B=\frac{(-2)^2+(-4)^2}{2}=\frac{20}{2}=10
\]

**Step 4 — Gradients**

\[
\frac{\partial J_B}{\partial m}
=\frac{2}{2}[(-2)(1)+(-4)(2)]=-10
\]

\[
\frac{\partial J_B}{\partial c}
=\frac{2}{2}[-2-4]=-6
\]

**Step 5 — Update**

\[
m_{\text{new}}=0-0.1(-10)=1.0
\]

\[
c_{\text{new}}=0-0.1(-6)=0.6
\]

Both examples contributed to one averaged gradient, and the parameters were updated once.

In [ ]:
# Pure Python only: manually calculate one mini-batch update
x_batch = [1.0, 2.0]
y_batch = [2.0, 4.0]
m, c = 0.0, 0.0
lr = 0.1
B = len(x_batch)

predictions = [m*x + c for x in x_batch]
errors = [p - y for p, y in zip(predictions, y_batch)]
mse = sum(e*e for e in errors) / B
grad_m = (2/B) * sum(e*x for e, x in zip(errors, x_batch))
grad_c = (2/B) * sum(errors)

m_new = m - lr*grad_m
c_new = c - lr*grad_c

print("Step 1 predictions:", predictions)
print("Step 2 errors:", errors)
print("Step 3 squared errors:", [e*e for e in errors])
print("Step 4 batch MSE:", mse)
print("Step 5 gradient for slope:", grad_m)
print("Step 6 gradient for intercept:", grad_c)
print("Step 7 new slope:", m_new)
print("Step 8 new intercept:", c_new)

## 6. Generate a dataset with a real learnable relationship

The article's example implementations use random input features and random binary labels. That is useful for demonstrating the mechanics of a mini-batch loop and recording loss, but independent random labels do not contain a stable input-to-target rule for a model to learn.

For a meaningful convergence comparison, we will instead generate synthetic regression data from:

\[
\boxed{y=4+3X+\text{noise}}
\]

The true intercept is 4 and the true slope is 3. The noise prevents the data points from lying perfectly on a line.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

SEED = 42
rng = np.random.default_rng(SEED)

X = 2 * rng.random((1000, 1))
y = 4 + 3 * X + rng.normal(0, 0.7, size=(1000, 1))

print("X shape:", X.shape, "| y shape:", y.shape)
print("True intercept = 4; true slope = 3")
display(pd.DataFrame({
    "X": X[:8, 0],
    "y": y[:8, 0]
}).round(4))

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(X[:, 0], y[:, 0], s=16, alpha=0.55, label="Generated samples")
x_line = np.linspace(0, 2, 200)
plt.plot(x_line, 4 + 3*x_line, color="red", linewidth=2, label="True line: y = 4 + 3X")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic regression dataset")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Mini-batch gradient descent with NumPy

We'll build the loop from scratch.

For each epoch:
1. Shuffle the training indices.
2. Split the shuffled examples into mini-batches.
3. Predict for the current batch.
4. Compute the batch errors.
5. Average the gradients over the **actual size of the batch**.
6. Update the parameters.
7. Calculate the full-dataset MSE after the epoch.

**Important implementation detail:** the final batch can contain fewer examples than the requested batch size. The gradient denominator must be `len(X_batch)`, not the nominal `batch_size`.

In [ ]:
def mini_batch_gd(X, y, learning_rate=0.01, epochs=50, batch_size=32, seed=42):
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).reshape(-1, 1)

    if len(X) != len(y):
        raise ValueError("X and y must have the same number of examples.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    n = len(X)
    X_bias = np.concatenate([np.ones((n, 1)), X], axis=1)
    theta = np.zeros((2, 1), dtype=float)  # [intercept, slope]
    rng = np.random.default_rng(seed)

    epoch_losses = []
    batch_losses = []
    parameter_history = []

    for epoch in range(epochs):
        order = rng.permutation(n)
        X_shuffled = X_bias[order]
        y_shuffled = y[order]

        for start in range(0, n, batch_size):
            X_batch = X_shuffled[start:start + batch_size]
            y_batch = y_shuffled[start:start + batch_size]
            actual_batch_size = len(X_batch)

            # Forward pass
            y_pred_batch = X_batch @ theta
            error = y_pred_batch - y_batch

            # MSE and mean gradient on this batch
            batch_loss = np.mean(error**2)
            gradient = (2 / actual_batch_size) * (X_batch.T @ error)

            # Parameter update
            theta -= learning_rate * gradient
            batch_losses.append(batch_loss)

        # Consistent full-dataset loss after each epoch
        full_predictions = X_bias @ theta
        epoch_loss = np.mean((full_predictions - y)**2)
        epoch_losses.append(epoch_loss)
        parameter_history.append(theta.ravel().copy())

    return {
        "theta": theta,
        "epoch_losses": np.array(epoch_losses),
        "batch_losses": np.array(batch_losses),
        "parameter_history": np.array(parameter_history)
    }

result = mini_batch_gd(X, y, learning_rate=0.01, epochs=50, batch_size=32, seed=42)
print("Learned intercept:", result["theta"][0, 0])
print("Learned slope:", result["theta"][1, 0])
print("Initial monitored MSE:", result["epoch_losses"][0])
print("Final monitored MSE:", result["epoch_losses"][-1])
print("Number of mini-batch updates:", len(result["batch_losses"]))

## 8. Visualize epoch loss and batch loss

The **epoch loss** is computed over all 1,000 examples after each epoch, so it gives a consistent comparison over time.

The **batch loss** is calculated on the current mini-batch before the update. It may fluctuate because different batches contain different examples. Batch loss values are therefore not expected to decrease monotonically.

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(np.arange(1, len(result["epoch_losses"]) + 1), result["epoch_losses"], marker=".")
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE after epoch")
plt.title("Epoch-level loss")
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(result["batch_losses"], alpha=0.75)
plt.xlabel("Mini-batch update")
plt.ylabel("Current mini-batch MSE before update")
plt.title("Batch-level loss (noisy)")
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 9. Plot the learned line

The model is trained to recover the generating relationship. Because the targets include noise, estimates will usually be near—not exactly equal to—the true slope 3 and intercept 4.

In [ ]:
theta = result["theta"].ravel()
x_grid = np.linspace(X.min(), X.max(), 200)
y_fitted = theta[0] + theta[1]*x_grid
y_true_line = 4 + 3*x_grid

plt.figure(figsize=(8, 4.5))
plt.scatter(X[:, 0], y[:, 0], s=14, alpha=0.4, label="Data")
plt.plot(x_grid, y_true_line, linestyle="--", linewidth=2, label="True line")
plt.plot(x_grid, y_fitted, linewidth=2, label="Mini-batch GD fit")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Regression line fitted using mini-batch gradient descent")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print(f"Fitted: y = {theta[0]:.4f} + {theta[1]:.4f}X")
print("True:   y = 4 + 3X")

## 10. Compare full-batch, strict SGD, and mini-batch GD fairly

The same dataset and MSE objective will be used for all three methods.

- Full-batch: batch size = the entire dataset.
- Strict SGD: batch size = 1.
- Mini-batch: batch size = 32.

All methods perform one pass over all training samples per epoch, but the number of parameter updates differs. For example, with \(N=1000\), strict SGD makes 1,000 updates per epoch, while batch GD makes one update per epoch.

In [ ]:
methods = [
    ("Full batch", len(X)),
    ("Mini-batch 32", 32),
    ("Mini-batch 128", 128),
    ("Strict SGD", 1),
]

method_results = {}
for name, bs in methods:
    method_results[name] = mini_batch_gd(
        X, y, learning_rate=0.01, epochs=50, batch_size=bs, seed=42
    )
    final_theta = method_results[name]["theta"].ravel()
    print(
        f"{name:>14}: intercept={final_theta[0]:.3f}, "
        f"slope={final_theta[1]:.3f}, "
        f"final MSE={method_results[name]['epoch_losses'][-1]:.4f}, "
        f"updates/epoch={(len(X)+bs-1)//bs}"
    )

In [ ]:
plt.figure(figsize=(9, 4.5))
for name, _ in methods:
    losses = method_results[name]["epoch_losses"]
    plt.plot(losses, label=name)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Convergence comparison: same data and objective")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 11. Study how batch size affects training

A small batch often produces noisier gradient estimates but more updates per epoch. A larger batch often produces less noisy estimates per update, but uses more memory and makes fewer updates per epoch.

Neither the smallest nor the largest batch size is universally best. Results depend on the model, hardware, learning rate, and dataset.

We will try batch sizes 8, 32, 128, and the full dataset while keeping the data, number of epochs, and learning rate fixed.

In [ ]:
batch_size_results = {}
for bs in [8, 32, 128, len(X)]:
    name = f"batch={bs}"
    batch_size_results[name] = mini_batch_gd(
        X, y, learning_rate=0.01, epochs=40, batch_size=bs, seed=123
    )

plt.figure(figsize=(9, 4.5))
for name, out in batch_size_results.items():
    plt.plot(out["epoch_losses"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Effect of batch size")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

rows = []
for name, out in batch_size_results.items():
    bs = int(name.split("=")[1])
    rows.append({
        "Batch size": bs,
        "Updates per epoch": (len(X) + bs - 1) // bs,
        "Final full-data MSE": out["epoch_losses"][-1],
        "Intercept estimate": out["theta"][0, 0],
        "Slope estimate": out["theta"][1, 0]
    })
display(pd.DataFrame(rows).sort_values("Batch size"))

## 12. Study the learning rate

The learning rate \(\eta\) controls the size of each update.

- Too small: slow progress.
- Suitable: the loss usually falls efficiently.
- Too large: updates may overshoot, oscillate, or diverge.

We'll vary learning rate while holding the batch size and data fixed. If the loss becomes non-finite, that is a failure to record—not something to hide by replacing values.

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.2]
lr_results = {}
for lr in learning_rates:
    lr_results[lr] = mini_batch_gd(
        X, y, learning_rate=lr, epochs=35, batch_size=32, seed=42
    )
    losses = lr_results[lr]["epoch_losses"]
    print(f"lr={lr:<5} final MSE={losses[-1]:.4g}; all finite={np.isfinite(losses).all()}")

plt.figure(figsize=(9, 4.5))
for lr, out in lr_results.items():
    losses = out["epoch_losses"]
    plt.plot(np.where(np.isfinite(losses), losses, np.nan), label=f"lr={lr}")
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Learning-rate comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 13. TensorFlow/Keras implementation

The article demonstrates mini-batch learning by setting `batch_size=32` in `model.fit()`. It also uses a callback to record batch loss.

Below, we use a learnable regression dataset instead of independent random labels so the loss has a meaningful relationship to learn. The callback records Keras's reported batch loss; this is the running loss metric for the training batches in the current epoch, so the curve may not equal each individual batch's standalone loss exactly.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

X_tf = X.astype("float32")
y_tf = y.astype("float32")

tf_model = models.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(1)
])
tf_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)

class BatchLossHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.batch_losses = []
        self.epoch_losses = []

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        if "loss" in logs:
            self.batch_losses.append(float(logs["loss"]))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.epoch_losses.append(float(logs.get("loss", np.nan)))

batch_history = BatchLossHistory()
tf_history = tf_model.fit(
    X_tf, y_tf,
    epochs=30,
    batch_size=32,
    shuffle=True,
    callbacks=[batch_history],
    verbose=0
)

print("Learned weights:", tf_model.get_weights())
print("Final epoch loss:", tf_history.history["loss"][-1])
print("Batch-loss records:", len(batch_history.batch_losses))

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(batch_history.batch_losses, alpha=0.7)
plt.xlabel("Mini-batch update")
plt.ylabel("Reported batch loss")
plt.title("Keras mini-batch loss history")
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(batch_history.epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Epoch training loss")
plt.title("Keras epoch loss")
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 14. PyTorch implementation

In PyTorch, a `DataLoader` forms mini-batches and shuffles examples. During each mini-batch update, the code runs a forward pass, computes MSE, clears old gradients, calls `backward()`, and steps the optimizer.

We use `batch_size=32`, matching the mini-batch size used in the article's code examples.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
X_pt = torch.tensor(X, dtype=torch.float32)
y_pt = torch.tensor(y, dtype=torch.float32)

train_dataset = TensorDataset(X_pt, y_pt)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

pt_model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(pt_model.parameters(), lr=0.01)

pt_batch_losses = []
pt_epoch_losses = []

for epoch in range(30):
    pt_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        predictions = pt_model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimizer.step()
        pt_batch_losses.append(loss.item())

    pt_model.eval()
    with torch.no_grad():
        full_mse = criterion(pt_model(X_pt), y_pt).item()
    pt_epoch_losses.append(full_mse)

print("PyTorch intercept:", pt_model.bias.item())
print("PyTorch slope:", pt_model.weight.item())
print("Final full-dataset MSE:", pt_epoch_losses[-1])

In [ ]:
plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(pt_batch_losses, alpha=0.7)
plt.xlabel("Mini-batch update")
plt.ylabel("Mini-batch MSE")
plt.title("PyTorch batch losses")
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(pt_epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("PyTorch epoch loss")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 15. Understand the article's random-label classification demo

The source article also demonstrates mini-batch training with:
- Random input features.
- Random binary labels.
- A Dense neural network ending in a sigmoid.
- Binary cross-entropy.
- Batch size 32 and per-batch loss recording, in both TensorFlow and PyTorch.

This is useful for teaching the **training loop mechanics**, but the inputs and labels are independently random, so there is no stable rule for the model to discover. With roughly balanced random labels, binary cross-entropy may hover near the random-prediction baseline rather than steadily approach a low value. For learning convergence concepts, the learnable regression example above is more informative.

A compact version of the article's pattern is:
```python
# TensorFlow/Keras pattern
model.fit(X_train, y_train, batch_size=32, epochs=10)
```
The equivalent PyTorch pattern is a `DataLoader(batch_size=32)` and one optimizer update per batch.

## 16. Advantages and limitations

### Advantages
- **Computational efficiency:** each update uses a subset rather than the entire dataset.
- **Memory efficiency:** useful for datasets that do not fit comfortably into one large operation.
- **More frequent updates than full-batch GD:** parameters update several times per epoch.
- **Hardware parallelism:** matrix operations on mini-batches typically work efficiently on GPUs and TPUs.
- **Less noisy than one-sample SGD in many cases:** averaging several examples can stabilize gradient estimates.

### Limitations
- Batch size must be selected and may require tuning.
- Small batches can produce noisy updates.
- Large batches use more memory and provide fewer updates per epoch.
- Multiple mini-batches must be managed, shuffled, and tracked.
- No single batch size guarantees the best result on every problem.

## 17. Common mistakes checklist

1. **Dividing by the configured batch size instead of the actual batch size:** the final batch may be smaller. Use `len(X_batch)`.
2. **Updating at the wrong time:** update once after each mini-batch, not after every example if you intend mini-batch GD.
3. **Not shuffling:** randomize sample order each epoch for the common training setup.
4. **Mixing batch loss and epoch loss:** batch loss is noisy; epoch-level full-data loss is more consistent for comparing training progress.
5. **Using random labels to judge learning:** independent random labels do not contain a learnable relationship.
6. **Learning rate too large:** can cause oscillation, unstable loss, or divergence.
7. **Wrong gradient sign:** if error is `prediction - target`, use gradients based on that sign consistently.
8. **Comparing algorithms with different conditions:** keep data, loss, learning rate, epoch count, and preprocessing aligned.

## 18. Knowledge check

1. With \(N=1000\) and batch size 32, how many updates happen per epoch if the final partial batch is kept?
2. Why does mini-batch gradient descent usually produce less noisy gradients than strict SGD?
3. How is mini-batch gradient descent different from full-batch GD?
4. Why should the gradient be averaged over the actual current batch size?
5. Why might batch-level loss fluctuate even while training is progressing?
6. Why are random inputs with independent random labels a weak dataset for demonstrating learning?
7. What does the learning rate control?

In [ ]:
N, B = 1000, 32
print("Q1 updates per epoch:", (N + B - 1)//B)
print("Q2: It averages gradient information from several examples.")
print("Q3: Mini-batch uses a subset per update; full batch uses every training example.")
print("Q4: The final batch can be smaller than the configured batch size.")
print("Q5: Different batches have different examples and therefore different losses.")
print("Q6: The labels contain no stable input-to-target relationship.")
print("Q7: It controls the size of each parameter update.")

## 19. Sources and official documentation

### Main article
- [GeeksforGeeks — Mini-Batch Gradient Descent in Deep Learning](https://www.geeksforgeeks.org/deep-learning/mini-batch-gradient-descent-in-deep-learning/) — mini-batch definition, update rule, how it works, batch-size guidance, learning-rate schedulers, momentum/Adam, TensorFlow and PyTorch examples, advantages, disadvantages, and quiz.

### Related reading
- [GeeksforGeeks — Mini-Batch Gradient Descent with Python](https://www.geeksforgeeks.org/machine-learning/ml-mini-batch-gradient-descent-with-python/)
- [GeeksforGeeks — Batch Size in Neural Network](https://www.geeksforgeeks.org/deep-learning/batch-size-in-neural-network/)
- [GeeksforGeeks — Different Variants of Gradient Descent](https://www.geeksforgeeks.org/machine-learning/different-variants-of-gradient-descent/)

### Official documentation
- [PyTorch `DataLoader`](https://docs.pytorch.org/docs/stable/data.html)
- [PyTorch SGD optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD)
- [TensorFlow/Keras `Model.fit`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit)
- [Keras SGD optimizer](https://keras.io/api/optimizers/sgd/)
- [NumPy random permutation](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.permutation.html)
- [NumPy mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

The notebook summarizes the article in original wording rather than copying the complete webpage. Official documentation is linked for checking APIs and troubleshooting.